In [ ]:
import os, subprocess
from pathlib import Path
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["OMP_NUM_THREADS"]="1"
work=Path("/kaggle/working/training");work.mkdir(exist_ok=True)
sources={'bluff_policy.py': '"""Compact learned Bluff action scorer using own cards and public evidence."""\nfrom collections import Counter, defaultdict\nimport json\nfrom pathlib import Path\n\nRANKS = ("A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K")\nPLACE_FEATURES = ["batch_size", "truthful", "bluff", "bluff_pile", "bluff_call_rate",\n                  "bluff_finish", "remaining_groups", "remaining_largest_group", "claimed_rank_held"]\nCHALLENGE_FEATURES = ["bias", "revealed_bluff_rate", "rank_held", "claim_count", "pile_size",\n                      "opponent_finishing", "own_small_hand", "opponent_small_hand", "samples"]\nINITIAL_WEIGHTS = [4., 5., -2., -4., -8., -8., -1., .2, .5,\n                   -3.5, 7., .7, -.3, -.5, 1., -1., .5, 0.]\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(INITIAL_WEIGHTS):\n            raise ValueError("wrong feature count")\n        self.weights = [float(v) for v in weights]\n        self.processed = 0\n        self.claims = {}\n        self.stats = defaultdict(lambda: [0, 0])\n        self.our_outcomes = [0, 0]\n\n    def update(self, observation):\n        history = observation.get("public_history", [])\n        if len(history) < self.processed:\n            self.processed, self.claims, self.stats, self.our_outcomes = 0, {}, defaultdict(lambda: [0, 0]), [0, 0]\n        for event in history[self.processed:]:\n            kind = event.get("type")\n            identity = event.get("claim_id")\n            if kind == "claim":\n                self.claims[identity] = (event.get("seat"), event.get("count", 1))\n            elif kind in ("claim_accepted", "challenge_resolved"):\n                placer, count = self.claims.pop(identity, (event.get("placer", event.get("seat")), event.get("count", 1)))\n                if placer == observation["seat"]:\n                    self.our_outcomes[0] += 1\n                    self.our_outcomes[1] += int(kind == "challenge_resolved")\n                if kind == "challenge_resolved" and isinstance(event.get("truthful"), bool):\n                    stats = self.stats[(placer, count)]\n                    stats[0] += 1\n                    stats[1] += int(not event["truthful"])\n        self.processed = len(history)\n\n    def candidates(self, observation):\n        hand = observation["hand"]\n        held = Counter(c["rank"] for c in hand)\n        pile = min(observation.get("pile_size", 0), 40) / 20.\n        if observation["phase"] == "CHALLENGE":\n            claim = observation["pending_claim"]\n            count, rank, placer = claim["count"], claim["claimed_rank"], claim["seat"]\n            copies = 4 * observation.get("game_config", {}).get("decks", 1)\n            if count > copies - held[rank]:\n                return [({"type": "challenge"}, [0.] * len(self.weights))]\n            samples, bluffs = self.stats[(placer, count)]\n            posterior = (bluffs + .6) / (samples + 5.)\n            sizes = observation["hand_sizes"]\n            vector = [1., posterior, held[rank] / copies, count / 4., pile,\n                      float(claim.get("provisional_finish", False)), float(len(hand) <= 3),\n                      1. / max(1, sizes[placer]), min(samples, 20) / 20.]\n            return [({"type": "accept"}, [0.] * len(self.weights)),\n                    ({"type": "challenge"}, [0.] * len(PLACE_FEATURES) + vector)]\n        groups = defaultdict(list)\n        for card in hand:\n            groups[card["rank"]].append(card)\n        actions = []\n        # Safe truthful groups are always available. Constructed bluff candidates\n        # use only currently owned references; the count is the actual batch size.\n        for rank, cards in groups.items():\n            actions.append({"type": "place", "card_refs": [c["card_ref"] for c in cards[:4]], "claimed_rank": rank})\n        ordered = sorted(hand, key=lambda c: (held[c["rank"]], RANKS.index(c["rank"])))\n        for count in (2, 3):\n            if len(ordered) >= count:\n                for rank in groups:\n                    actions.append({"type": "place", "card_refs": [c["card_ref"] for c in ordered[:count]], "claimed_rank": rank})\n        call_rate = (self.our_outcomes[1] + 1.) / (self.our_outcomes[0] + 4.)\n        by_ref = {c["card_ref"]: c for c in hand}\n        result = []\n        for action in actions:\n            refs = set(action["card_refs"])\n            rank = action["claimed_rank"]\n            truthful = all(by_ref[ref]["rank"] == rank for ref in refs)\n            remaining = Counter(c["rank"] for c in hand if c["card_ref"] not in refs)\n            vector = [len(refs), float(truthful), float(not truthful), float(not truthful) * pile,\n                      float(not truthful) * call_rate, float(not truthful and len(refs) == len(hand)),\n                      len(remaining) / 13., max(remaining.values(), default=0) / 4., held[rank] / 4.]\n            result.append((action, vector + [0.] * len(CHALLENGE_FEATURES)))\n        return result\n\n    def act(self, observation):\n        self.update(observation)\n        return max(self.candidates(observation), key=lambda candidate:\n                   sum(w * value for w, value in zip(self.weights, candidate[1])))[0]\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({"format": "triad-bluff-linear-v1", "features": PLACE_FEATURES + CHALLENGE_FEATURES,\n            "weights": self.weights, "training": metadata}, indent=2) + "\\n")\n', 'bluff_policy_v2.py': '"""Learned Bluff scorer with conservative rank bounds and public caller evidence.\n\nInference reads only own hand/private history and public events. Unknown discarded\ncards decrement every rank bound, ensuring no hidden-hand identity is assumed.\n"""\nfrom collections import Counter, defaultdict\nimport json\nimport math\nfrom pathlib import Path\n\nfrom bluff_policy import Policy as V1, PLACE_FEATURES, CHALLENGE_FEATURES, INITIAL_WEIGHTS as V1_INITIAL, RANKS\n\nEXTRA_PLACE = [\'bluff_any_call_probability\', \'bluff_impossible_for_caller\',\n               \'truthful_turns_removed\', \'bluff_known_pile_fraction\',\n               \'truthful_finish\', \'bluff_no_impossibility_call_possible\']\nEXTRA_CHALLENGE = [\'placer_global_revealed_bluff_rate\', \'known_outside_rank_fraction\',\n                   \'known_pile_fraction\', \'claim_fraction_of_original_hand\',\n                   \'finishing_large_batch\', \'known_placer_rank_before_claim\']\nFEATURE_NAMES = PLACE_FEATURES + CHALLENGE_FEATURES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = [.5, 1., -.5, -2., -8., -3., -1., .2, .5] + V1_INITIAL[len(PLACE_FEATURES):] + [-6., -12., 3., -1., 4., 3., 1., 1., -.5, .5, .5, -.5]\n\n\ndef identity(card):\n    return card.get(\'deck\', 0), card[\'rank\'], card[\'suit\']\n\n\nclass Policy(V1):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid model weights\')\n        super().__init__(weights[:len(V1_INITIAL)])\n        self.weights = list(map(float, weights))\n        self._reset_evidence()\n\n    def _reset_evidence(self):\n        self.folded_history = []\n        self.known = defaultdict(Counter)\n        self.pile_known = {}\n        self.calls = defaultdict(lambda: [0, 0])\n        self.global_bluffs = defaultdict(lambda: [0, 0])\n        self.pending_meta = {}\n        self.active = []\n        self.before_claim = {}\n\n    def update(self, observation):\n        history = observation.get(\'public_history\', [])\n        if len(history) < len(self.folded_history) or history[:len(self.folded_history)] != self.folded_history:\n            self.processed = 0\n            self.claims = {}\n            self.stats = defaultdict(lambda: [0, 0])\n            self.our_outcomes = [0, 0]\n            self._reset_evidence()\n        super().update(observation)\n        own_places = {event[\'claim_id\']: event[\'cards\'] for event in observation.get(\'private_history\', []) if event.get(\'type\') == \'own_placement\'}\n        for event in history[len(self.folded_history):]:\n            kind, cid = event.get(\'type\'), event.get(\'claim_id\')\n            if kind == \'game_start\':\n                self.active = list(event[\'active_seats\'])\n            elif kind == \'claim\':\n                placer, count = event[\'seat\'], event[\'count\']\n                self.before_claim[cid] = dict(self.known[placer])\n                # The rank in an unverified claim is not evidence of actual cards.\n                for rank in list(self.known[placer]):\n                    self.known[placer][rank] = max(0, self.known[placer][rank] - count)\n                priority = sorted((s for s in self.active if s != placer), key=lambda s: (s - placer) % observation[\'game_config\'][\'players\'])\n                self.pending_meta[cid] = (placer, count, priority, bool(event.get(\'provisional_finish\')))\n                for card in own_places.get(cid, []):\n                    self.pile_known[identity(card)] = card\n            elif kind in (\'claim_accepted\', \'challenge_resolved\'):\n                placer, count, priority, finish = self.pending_meta.pop(cid)\n                selected = event.get(\'challenger\') if kind == \'challenge_resolved\' else None\n                # The first selected challenger is revealed. Earlier seats must\n                # have accepted; later choices remain unknown and are not counted.\n                for seat in priority:\n                    record = self.calls[(seat, count, finish)]\n                    record[0] += 1\n                    record[1] += int(seat == selected)\n                    if seat == selected:\n                        break\n                if kind == \'challenge_resolved\':\n                    record = self.global_bluffs[placer]\n                    record[0] += 1\n                    record[1] += int(not event[\'truthful\'])\n                    for card in event[\'revealed_batch\']:\n                        self.pile_known[identity(card)] = card\n            elif kind == \'pickup\':\n                recipient = event[\'recipient\']\n                self.known[recipient].update(card[\'rank\'] for card in self.pile_known.values())\n                self.pile_known = {}\n            elif kind == \'finish\':\n                self.active = [s for s in self.active if s != event[\'seat\']]\n        self.folded_history = list(history)\n\n    def caller_probability(self, observation, count, finish):\n        escape = 1.\n        for seat in observation[\'active_seats\']:\n            if seat == observation[\'seat\']:\n                continue\n            samples, calls = self.calls.get((seat, count, finish), (0, 0))\n            # Beta prior avoids certainties from small or selectively observed samples.\n            escape *= 1. - (calls + 1.) / (samples + 5.)\n        return 1. - escape\n\n    def candidates(self, observation):\n        hand = observation[\'hand\']\n        held = Counter(c[\'rank\'] for c in hand)\n        copies = 4 * observation.get(\'game_config\', {}).get(\'decks\', 1)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            claim = observation[\'pending_claim\']\n            placer, rank, count = claim[\'seat\'], claim[\'claimed_rank\'], claim[\'count\']\n            outside = held[rank] + sum(counts[rank] for seat, counts in self.known.items() if seat not in (observation[\'seat\'], placer))\n            if count > copies - outside:\n                return [({\'type\': \'challenge\'}, [0.] * len(self.weights))]\n            base = super().candidates(observation)\n            samples, bluffs = self.global_bluffs.get(placer, (0, 0))\n            extra = [(bluffs + .6) / (samples + 5.), outside / copies,\n                     len(self.pile_known) / max(1, observation[\'pile_size\']),\n                     count / max(1, observation[\'hand_sizes\'][placer] + count),\n                     float(claim.get(\'provisional_finish\', False)) * count / 4.,\n                     self.before_claim.get(claim[\'claim_id\'], {}).get(rank, 0) / copies]\n            return [(a, v + [0.] * len(EXTRA_PLACE) + (extra if a[\'type\'] == \'challenge\' else [0.] * len(EXTRA_CHALLENGE))) for a, v in base]\n        groups = defaultdict(list)\n        for card in hand:\n            groups[card[\'rank\']].append(card)\n        actions = [action for action, _ in super().candidates(observation)]\n        # Full truthful batches plus partial truthful batches and mixed shedding\n        # batches of up to four cards, including a possible finishing bluff.\n        for rank, cards in groups.items():\n            for count in range(1, min(4, len(cards)) + 1):\n                actions.append({\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in cards[:count]], \'claimed_rank\': rank})\n        ordered = sorted(hand, key=lambda c: (held[c[\'rank\']], RANKS.index(c[\'rank\'])))\n        for count in range(2, min(4, len(hand)) + 1):\n            for rank in RANKS:\n                actions.append({\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in ordered[:count]], \'claimed_rank\': rank})\n        by_ref, seen, result = {c[\'card_ref\']: c for c in hand}, set(), []\n        pile = min(observation.get(\'pile_size\', 0), 40) / 20.\n        call_rate = (self.our_outcomes[1] + 1.) / (self.our_outcomes[0] + 4.)\n        for action in actions:\n            refs, rank = set(action[\'card_refs\']), action[\'claimed_rank\']\n            key = tuple(sorted(refs)), rank\n            if key in seen:\n                continue\n            seen.add(key)\n            truthful = all(by_ref[r][\'rank\'] == rank for r in refs)\n            bluff = float(not truthful)\n            remaining = Counter(c[\'rank\'] for c in hand if c[\'card_ref\'] not in refs)\n            finish = not remaining\n            vector = [len(refs), float(truthful), bluff, bluff * pile, bluff * call_rate,\n                      float(bluff and finish), len(remaining) / 13., max(remaining.values(), default=0) / 4., held[rank] / 4.]\n            impossible = any(self.known.get(s, {}).get(rank, 0) + len(refs) > copies for s in observation[\'active_seats\'] if s != observation[\'seat\'])\n            extra = [bluff * self.caller_probability(observation, len(refs), finish),\n                     bluff * impossible, sum((n + 3) // 4 for n in held.values()) - sum((n + 3) // 4 for n in remaining.values()),\n                     bluff * len(self.pile_known) / max(1, observation[\'pile_size\']),\n                     float(truthful and finish), bluff * float(held[rank] >= len(refs))]\n            result.append((action, vector + [0.] * len(CHALLENGE_FEATURES) + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return result\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v2\', \'features\': FEATURE_NAMES,\n                                        \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_policy_v3.py': '"""Bluff v2 model with additional, sound public/own pile rank deductions.\n\nOlder unchallenged batches are never assumed truthful. These deductions use\nonly own placement cards and the last publicly revealed batch before pickup.\n"""\nfrom collections import Counter\nfrom bluff_policy_v2 import Policy as V2, FEATURE_NAMES, INITIAL_WEIGHTS\n\n\nclass Policy(V2):\n    def candidates(self, observation):\n        if observation[\'phase\'] == \'CHALLENGE\':\n            claim = observation[\'pending_claim\']\n            rank, count, placer = claim[\'claimed_rank\'], claim[\'count\'], claim[\'seat\']\n            # Known old pile cards, current own cards and lower bounds on other\n            # players\' hands are disjoint, so their sum is a conservative count.\n            pile_count = Counter(card[\'rank\'] for card in self.pile_known.values())[rank]\n            held_count = sum(card[\'rank\'] == rank for card in observation[\'hand\'])\n            other_count = sum(counts[rank] for seat, counts in self.known.items()\n                              if seat not in (observation[\'seat\'], placer))\n            copies = 4 * observation[\'game_config\'][\'decks\']\n            if count > copies - held_count - other_count - pile_count:\n                return [({\'type\': \'challenge\'}, [0.] * len(self.weights))]\n        return super().candidates(observation)\n', 'bluff_policy_v4.py': '"""Linear Bluff scorer with learned interactions for public call-risk adaptation."""\nfrom collections import Counter\nimport json\nimport math\nfrom pathlib import Path\nfrom bluff_policy_v3 import Policy as V3\nfrom bluff_policy_v2 import FEATURE_NAMES as V2_NAMES, INITIAL_WEIGHTS as V2_INITIAL\n\nEXTRA_PLACE = [\'bluff_call_probability_batch\', \'bluff_call_probability_pile\',\n               \'bluff_call_probability_finish\', \'bluff_nearest_opponent_finish\',\n               \'truthful_call_probability_batch\', \'bluff_remaining_truthful_turns\']\nEXTRA_CHALLENGE = [\'global_revealed_bluff_finishing\', \'count_revealed_bluff_pile\',\n                   \'global_revealed_bluff_pile\', \'count_revealed_bluff_small_own_hand\',\n                   \'global_revealed_bluff_count\', \'opponent_hand_pace_relative_to_own\']\nFEATURE_NAMES = V2_NAMES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = V2_INITIAL + [0.] * 12\n\n\nclass Policy(V3):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid v4 model weights\')\n        super().__init__(weights[:len(V2_NAMES)])\n        self.weights = list(map(float, weights))\n\n    def candidates(self, observation):\n        candidates = super().candidates(observation)\n        hand = observation[\'hand\']\n        pile = min(observation[\'pile_size\'], 40) / 20.\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if len(candidates) == 1:\n                return [(candidates[0][0], [0.] * len(self.weights))]\n            claim = observation[\'pending_claim\']\n            samples, bluffs = self.global_bluffs.get(claim[\'seat\'], (0, 0))\n            global_rate = (bluffs + .6) / (samples + 5.)\n            samples, bluffs = self.stats.get((claim[\'seat\'], claim[\'count\']), (0, 0))\n            count_rate = (bluffs + .6) / (samples + 5.)\n            extra = [global_rate * float(claim.get(\'provisional_finish\', False)),\n                     count_rate * pile, global_rate * pile,\n                     count_rate * float(len(hand) <= 3), global_rate * claim[\'count\'] / 4.,\n                     (len(hand) - observation[\'hand_sizes\'][claim[\'seat\']]) / max(1, len(hand))]\n            return [(action, vector[:len(V2_NAMES)] + [0.] * len(EXTRA_PLACE) +\n                     (extra if action[\'type\'] == \'challenge\' else [0.] * len(EXTRA_CHALLENGE)))\n                    for action, vector in candidates]\n        by_ref = {card[\'card_ref\']: card for card in hand}\n        held = Counter(card[\'rank\'] for card in hand)\n        nearest = min(observation[\'hand_sizes\'][seat] for seat in observation[\'active_seats\']\n                      if seat != observation[\'seat\'])\n        results = []\n        risks = {}\n        for action, vector in candidates:\n            count = len(action[\'card_refs\']); finish = count == len(hand)\n            if count not in risks:\n                risks[count] = self.caller_probability(observation, count, finish)\n            risk = risks[count]\n            bluff = float(any(by_ref[ref][\'rank\'] != action[\'claimed_rank\'] for ref in action[\'card_refs\']))\n            remaining = held.copy()\n            for ref in action[\'card_refs\']:\n                remaining[by_ref[ref][\'rank\']] -= 1\n            turns = sum((n + 3) // 4 for n in remaining.values() if n > 0)\n            extra = [bluff * risk * count / 4., bluff * risk * pile,\n                     bluff * risk * float(finish), bluff / max(1, nearest),\n                     (1. - bluff) * risk * count / 4., bluff * turns / 13.]\n            results.append((action, vector[:len(V2_NAMES)] + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return results\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v4\',\n            \'features\': FEATURE_NAMES, \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_policy_v5.py': '"""Public caller pooling and uncertainty features for a learned Bluff scorer.\n\nBeta summaries are modeling features, not calibrated certainty claims. Observed\nselected/earlier caller outcomes are pooled; later sealed votes stay unknown.\n"""\nimport json\nimport math\nfrom pathlib import Path\nfrom bluff_policy_v4 import Policy as V4, FEATURE_NAMES as V4_NAMES, INITIAL_WEIGHTS as V4_INITIAL\n\nEXTRA_PLACE = [\'bluff_pooled_caller_probability\', \'bluff_pooled_caller_uncertainty\',\n               \'bluff_specific_caller_uncertainty\', \'bluff_sparse_own_outcomes\',\n               \'truthful_pooled_caller_probability_batch\', \'bluff_pooled_caller_probability_pile\']\nEXTRA_CHALLENGE = [\'global_revealed_bluff_uncertainty\', \'count_revealed_bluff_uncertainty\']\nFEATURE_NAMES = V4_NAMES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = V4_INITIAL + [0.] * 8\n\n\ndef beta_summary(samples, successes, prior_success=1., prior_total=5.):\n    a = successes + prior_success\n    b = samples - successes + prior_total - prior_success\n    return a / (a + b), math.sqrt(a * b / ((a + b) ** 2 * (a + b + 1.)))\n\n\nclass Policy(V4):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid v5 model weights\')\n        super().__init__(weights[:len(V4_NAMES)])\n        self.weights = list(map(float, weights))\n\n    def caller_summaries(self, observation, count=None, finish=None):\n        escape_mean = escape_upper = 1.\n        for seat in observation[\'active_seats\']:\n            if seat == observation[\'seat\']:\n                continue\n            if count is None:\n                samples = calls = 0\n                for (caller, _, _), record in self.calls.items():\n                    if caller == seat:\n                        samples += record[0]; calls += record[1]\n            else:\n                samples, calls = self.calls.get((seat, count, finish), (0, 0))\n            mean, deviation = beta_summary(samples, calls)\n            escape_mean *= 1. - mean\n            escape_upper *= 1. - min(1., mean + deviation)\n        return 1. - escape_mean, escape_mean - escape_upper\n\n    def candidates(self, observation):\n        candidates = super().candidates(observation)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if len(candidates) == 1:\n                return [(candidates[0][0], [0.] * len(self.weights))]\n            claim = observation[\'pending_claim\']\n            samples, bluffs = self.global_bluffs.get(claim[\'seat\'], (0, 0))\n            _, global_uncertainty = beta_summary(samples, bluffs, .6, 5.)\n            samples, bluffs = self.stats.get((claim[\'seat\'], claim[\'count\']), (0, 0))\n            _, count_uncertainty = beta_summary(samples, bluffs, .6, 5.)\n            return [(action, vector[:len(V4_NAMES)] + [0.] * len(EXTRA_PLACE) +\n                     ([global_uncertainty, count_uncertainty] if action[\'type\'] == \'challenge\' else [0., 0.]))\n                    for action, vector in candidates]\n        by_ref = {card[\'card_ref\']: card for card in observation[\'hand\']}\n        pooled, pooled_uncertainty = self.caller_summaries(observation)\n        sparse = float(self.our_outcomes[0] <= 2)\n        pile = min(observation[\'pile_size\'], 40) / 20.\n        risks = {}\n        results = []\n        for action, vector in candidates:\n            count = len(action[\'card_refs\'])\n            if count not in risks:\n                risks[count] = self.caller_summaries(observation, count, count == len(observation[\'hand\']))\n            _, specific_uncertainty = risks[count]\n            bluff = float(any(by_ref[ref][\'rank\'] != action[\'claimed_rank\'] for ref in action[\'card_refs\']))\n            extra = [bluff * pooled, bluff * pooled_uncertainty, bluff * specific_uncertainty,\n                     bluff * sparse, (1. - bluff) * pooled * count / 4., bluff * pooled * pile]\n            results.append((action, vector[:len(V4_NAMES)] + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return results\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v5\',\n            \'features\': FEATURE_NAMES, \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_training_observation.py': '"""Training-only observation shortcut for verified read-only simulator actors.\n\nAll fields still come from the SDK\'s filtered observe(). The engine is not\nadvanced while histories are temporarily detached. Only the requested seat\'s\nprivate history is returned. Never import this helper into an inference image.\n"""\nfrom triad_bluff import engine\n\n\ndef observe_read_only(state, seat):\n    public = state.public_history\n    private = state.private_history[seat]\n    try:\n        state.public_history = []\n        state.private_history[seat] = []\n        observation = engine.observe(state, seat)\n    finally:\n        state.public_history = public\n        state.private_history[seat] = private\n    observation[\'public_history\'] = public\n    observation[\'private_history\'] = private\n    return observation\n', 'bluff_mlp_policy.py': '"""Tiny residual neural Bluff scorer with standard-library inference."""\nimport math\nfrom bluff_policy_v5 import Policy as Features, FEATURE_NAMES\n\n\nclass Policy(Features):\n    def __init__(self, document):\n        if document.get(\'format\') != \'triad-bluff-residual-mlp-v1\' or document.get(\'features\') != FEATURE_NAMES:\n            raise ValueError(\'incompatible neural Bluff schema\')\n        base = document[\'base_weights\']\n        super().__init__(base)\n        self.hidden = document[\'hidden_weight\']; self.bias = document[\'hidden_bias\']\n        self.output = document[\'output_weight\']; self.output_bias = float(document[\'output_bias\'])\n        self.means = document[\'feature_mean\']; self.scales = document[\'feature_scale\']\n        width = len(self.hidden)\n        if not 1 <= width <= 128 or len(self.bias) != width or len(self.output) != width:\n            raise ValueError(\'invalid hidden layer shape\')\n        if any(len(row) != len(FEATURE_NAMES) for row in self.hidden) or len(self.means) != len(FEATURE_NAMES) or len(self.scales) != len(FEATURE_NAMES):\n            raise ValueError(\'invalid neural input shape\')\n        parameters = base + self.bias + self.output + self.means + self.scales + [self.output_bias]\n        parameters += [weight for row in self.hidden for weight in row]\n        if not all(math.isfinite(float(weight)) for weight in parameters) or any(scale <= 0. for scale in self.scales):\n            raise ValueError(\'nonfinite neural parameters or invalid feature scale\')\n\n    def score_features(self, features):\n        normalized = [(value - mean) / scale for value, mean, scale in zip(features, self.means, self.scales)]\n        residual = self.output_bias\n        for row, bias, output in zip(self.hidden, self.bias, self.output):\n            hidden = max(0., bias + sum(weight * value for weight, value in zip(row, normalized)))\n            residual += output * hidden\n        return sum(weight * value for weight, value in zip(self.weights, features)) + residual\n\n    def act(self, observation):\n        self.update(observation)\n        return max(self.candidates(observation), key=lambda item: self.score_features(item[1]))[0]\n\n\nclass FastPolicy(Policy):\n    """Training-only NumPy batching; near ties use exact deployment scoring."""\n    def __init__(self, document):\n        super().__init__(document)\n        import numpy as np\n        self.np = np\n        self.matrix = np.asarray(self.hidden, dtype=np.float64)\n        self.hidden_bias = np.asarray(self.bias, dtype=np.float64)\n        self.output_array = np.asarray(self.output, dtype=np.float64)\n        self.base_array = np.asarray(self.weights, dtype=np.float64)\n        self.mean_array = np.asarray(self.means, dtype=np.float64)\n        self.scale_array = np.asarray(self.scales, dtype=np.float64)\n\n    def act(self, observation):\n        self.update(observation)\n        candidates = self.candidates(observation)\n        matrix = self.np.asarray([vector for _, vector in candidates], dtype=self.np.float64)\n        scores = matrix @ self.base_array + self.output_bias\n        scores += self.np.maximum(0., ((matrix - self.mean_array) / self.scale_array) @ self.matrix.T + self.hidden_bias) @ self.output_array\n        largest = float(scores.max())\n        band = 1e-8 * max(1., abs(largest))\n        finalists = [index for index, value in enumerate(scores) if value >= largest - band]\n        chosen = max(finalists, key=lambda index: self.score_features(candidates[index][1]))\n        return candidates[chosen][0]\n', 'train_bluff_league.py': '"""Train against diverse Bluff policies with paired seat blocks and held-out tests."""\nimport argparse\nimport concurrent.futures\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport numpy as np\n\nROOT = Path(__file__).resolve().parent\nSDK = Path(os.environ.get(\'BLUFF_SDK\', ROOT.parent / \'bluffSDK\'))\nsys.path.insert(0, str(SDK / \'local_evaluator/src\'))\nfrom triad_bluff import engine\nfrom triad_bluff.baselines import HeuristicAgent, RandomAgent, TruthfulAgent, AggressiveChallengeAgent\nfrom bluff_policy import Policy as V1, INITIAL_WEIGHTS as V1_INITIAL, RANKS\nfrom bluff_policy_v2 import Policy as V2, INITIAL_WEIGHTS, FEATURE_NAMES\nfrom bluff_policy_v3 import Policy as V3\nfrom bluff_policy_v4 import Policy as V4, INITIAL_WEIGHTS as V4_INITIAL\nfrom bluff_policy_v5 import Policy as V5, INITIAL_WEIGHTS as V5_INITIAL\nfrom bluff_training_observation import observe_read_only\n\n\nclass Starter(TruthfulAgent):\n    def act(self, observation):\n        if observation[\'phase\'] == \'CHALLENGE\':\n            return super().act(observation)\n        from collections import defaultdict\n        groups = defaultdict(list)\n        for card in observation[\'hand\']:\n            groups[card[\'rank\']].append(card)\n        rank, cards = max(groups.items(), key=lambda item: (len(item[1]), item[0]))\n        return {\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in cards[:4]], \'claimed_rank\': rank}\n\n\nclass SDKCompetitive:\n    """The SDK\'s supplied adaptive strategy, used only as a development rival."""\n    def __init__(self):\n        import importlib.util\n        path = SDK / \'agents/bluff-agent/src/strategy.py\'\n        spec = importlib.util.spec_from_file_location(\'sdk_competitive_strategy\', path)\n        module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(module)\n        self.select_action = module.select_action\n        self.state = None\n\n    def act(self, observation):\n        action, self.state = self.select_action(observation, self.state)\n        return action\n\n\nclass StochasticBluffer(TruthfulAgent):\n    def __init__(self, seed, bluff_rate, call_rate):\n        self.rng = random.Random(seed)\n        self.bluff_rate, self.call_rate = bluff_rate, call_rate\n\n    def act(self, observation):\n        truthful = super().act(observation)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if truthful[\'type\'] == \'challenge\':\n                return truthful\n            finishing = observation[\'pending_claim\'].get(\'provisional_finish\', False)\n            probability = min(.95, self.call_rate + .2 * finishing)\n            return {\'type\': \'challenge\' if self.rng.random() < probability else \'accept\'}\n        hand = observation[\'hand\']\n        if len(hand) > 1 and self.rng.random() < self.bluff_rate:\n            from collections import Counter\n            held = Counter(c[\'rank\'] for c in hand)\n            count = min(4, len(hand)) if len(hand) <= 4 else self.rng.choice((2, 3, 4))\n            chosen = sorted(hand, key=lambda c: (held[c[\'rank\']], RANKS.index(c[\'rank\'])))[:count]\n            # Owning copies of the claimed rank makes impossible calls less likely.\n            claimed = max(RANKS, key=lambda rank: held[rank])\n            return {\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in chosen], \'claimed_rank\': claimed}\n        return truthful\n\n\ndef make_policy(spec, seed):\n    kind = spec[\'kind\']\n    if kind == \'mlp\':\n        from bluff_mlp_policy import FastPolicy\n        return FastPolicy(spec[\'model\'])\n    if kind == \'v1\':\n        return V1(spec[\'weights\'])\n    if kind == \'v2\':\n        return V2(spec[\'weights\'])\n    if kind == \'v3\':\n        return V3(spec[\'weights\'])\n    if kind == \'v4\':\n        return V4(spec[\'weights\'])\n    if kind == \'v5\':\n        return V5(spec[\'weights\'])\n    if kind == \'bluffer\':\n        return StochasticBluffer(seed, spec[\'bluff_rate\'], spec[\'call_rate\'])\n    if kind == \'starter\':\n        return Starter()\n    if kind == \'competitive\':\n        return SDKCompetitive()\n    if kind == \'truthful\':\n        return TruthfulAgent()\n    if kind == \'random\':\n        return RandomAgent(seed)\n    if kind == \'aggressive\':\n        return AggressiveChallengeAgent(seed)\n    return HeuristicAgent(seed)\n\n\ndef score_game(seed, seat, candidate, rivals, horizon=1000):\n    actors = [None] * 4\n    actors[seat] = make_policy(candidate, seed * 10 + 9)\n    for offset, rival in enumerate(rivals, 1):\n        actors[(seat + offset) % 4] = make_policy(rival, seed * 10 + offset)\n    state = engine.new_game(engine.GameConfig(max_turns=horizon), seed)\n    while state.outcome is None and seat not in state.finished_set:\n        decision = engine.next_decision(state)\n        observation = observe_read_only(state, decision.seat) if os.environ.get(\'BLUFF_FAST_OBSERVATIONS\') == \'1\' else engine.observe(state, decision.seat)\n        payload = actors[decision.seat].act(observation)\n        action = engine.decode_action(state, decision, payload)\n        if not engine.step(state, decision, action).ok:\n            raise RuntimeError(\'policy produced an illegal action\')\n    if seat in state.finished_order:\n        placement = state.finished_order.index(seat)\n        return state.config.position_points[placement], float(placement == 0)\n    return state.outcome.points[seat], 0.\n\n\ndef evaluate_job(job):\n    candidate, seeds, pool, mode, *options = job\n    horizon = options[0] if options else 1000\n    points, wins = [], []\n    for seed in seeds:\n        rng = random.Random(seed ^ 0x424C554646)\n        if mode == \'league\':\n            rivals = [rng.choice(pool) for _ in range(3)]\n        elif mode == \'bluffer\':\n            rivals = [{\'kind\': \'bluffer\', \'bluff_rate\': .35, \'call_rate\': .1}] * 3\n        else:\n            rivals = [{\'kind\': mode}] * 3\n        outcomes = [score_game(seed, seat, candidate, rivals, horizon) for seat in range(4)]\n        points.append(sum(p for p, _ in outcomes) / 4.)\n        wins.append(sum(w for _, w in outcomes) / 4.)\n    return {\'points\': float(np.mean(points)), \'win_rate\': float(np.mean(wins)),\n            \'games\': 4 * len(seeds), \'block_points\': points, \'block_wins\': wins}\n\n\ndef summarize(result):\n    return {k: v for k, v in result.items() if not k.startswith(\'block_\')}\n\n\ndef parallel_evaluate(pool, candidate, seeds, reference, mode):\n    """Distribute validation seed blocks instead of leaving other workers idle."""\n    chunks = [seeds[i:i + 16] for i in range(0, len(seeds), 16)]\n    parts = list(pool.map(evaluate_job, [(candidate, chunk, reference, mode) for chunk in chunks]))\n    points = [point for part in parts for point in part[\'block_points\']]\n    wins = [win for part in parts for win in part[\'block_wins\']]\n    return {\'points\': float(np.mean(points)), \'win_rate\': float(np.mean(wins)),\n            \'games\': 4 * len(points), \'block_points\': points, \'block_wins\': wins}\n\n\ndef paired_interval(candidate, reference, seed=20261014):\n    differences = np.asarray(candidate[\'block_points\']) - np.asarray(reference[\'block_points\'])\n    rng = np.random.default_rng(seed)\n    means = differences[rng.integers(0, len(differences), (5000, len(differences)))].mean(axis=1)\n    return {\'mean_gain\': float(differences.mean()), \'ci95\': [float(v) for v in np.quantile(means, [.025, .975])],\n            \'unit\': \'points per game\', \'sampling_unit\': \'four-seat seed block\'}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--incumbent\', type=Path)\n    parser.add_argument(\'--reference-policy\', type=Path)\n    parser.add_argument(\'--policy-kind\', choices=(\'v2\', \'v3\', \'v4\', \'v5\'), default=\'v2\')\n    parser.add_argument(\'--fast-observations\', action=\'store_true\')\n    parser.add_argument(\'--stress-opponents\', action=\'store_true\')\n    parser.add_argument(\'--selection-seed-start\', type=int, default=8_000_000)\n    parser.add_argument(\'--test-seed-start\', type=int, default=9_000_000)\n    parser.add_argument(\'--train-seed-start\', type=int, default=50_000)\n    parser.add_argument(\'--iterations\', type=int, default=30)\n    parser.add_argument(\'--population\', type=int, default=32)\n    parser.add_argument(\'--train-blocks\', type=int, default=8)\n    parser.add_argument(\'--train-horizon\', type=int, default=200)\n    parser.add_argument(\'--validation-blocks\', type=int, default=64)\n    parser.add_argument(\'--test-blocks\', type=int, default=256)\n    parser.add_argument(\'--workers\', type=int, default=min(4, os.cpu_count() or 1))\n    parser.add_argument(\'--seed\', type=int, default=20261016)\n    parser.add_argument(\'--output\', type=Path, default=ROOT / \'bluff-league-output\')\n    args = parser.parse_args()\n    if min(args.iterations, args.population, args.train_blocks, args.validation_blocks, args.test_blocks, args.workers, args.train_horizon) < 1 or args.train_horizon > 1000:\n        parser.error(\'counts must be positive\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    if args.fast_observations:\n        os.environ[\'BLUFF_FAST_OBSERVATIONS\'] = \'1\'\n    incumbent_document = json.loads(args.incumbent.read_text()) if args.incumbent else {\'weights\': V1_INITIAL}\n    weights = incumbent_document[\'weights\']\n    incumbent_kind = incumbent_document.get(\'training\', {}).get(\'policy_kind\', \'v2\' if len(weights) == len(FEATURE_NAMES) else \'v4\' if len(weights) == len(V4_INITIAL) else \'v5\' if len(weights) == len(V5_INITIAL) else \'v1\')\n    incumbent = {\'kind\': incumbent_kind, \'weights\': weights}\n    initial_weights = V5_INITIAL if args.policy_kind == \'v5\' else V4_INITIAL if args.policy_kind == \'v4\' else INITIAL_WEIGHTS\n    mean = np.asarray(weights + initial_weights[len(weights):], dtype=float)\n    best, sigma = mean.copy(), np.full(len(mean), 1.5)\n    rng = np.random.default_rng(args.seed)\n    cautious = list(INITIAL_WEIGHTS); cautious[2] -= 8.; cautious[4] -= 8.\n    daring = list(INITIAL_WEIGHTS); daring[2] += 4.; daring[5] += 4.\n    reference = [{\'kind\': k} for k in (\'starter\', \'heuristic\', \'truthful\', \'aggressive\', \'random\', \'competitive\')] + [\n        incumbent, {\'kind\': \'v2\', \'weights\': INITIAL_WEIGHTS}, {\'kind\': \'v2\', \'weights\': cautious},\n        {\'kind\': \'v2\', \'weights\': daring}] + [\n        {\'kind\': \'bluffer\', \'bluff_rate\': b, \'call_rate\': c} for b,c in ((.15,.03),(.35,.1),(.5,.35))]\n    if args.reference_policy:\n        reference_document = json.loads(args.reference_policy.read_text())\n        reference.append({\'kind\': reference_document.get(\'training\', {}).get(\'policy_kind\', \'v1\' if len(reference_document[\'weights\']) == len(V1_INITIAL) else \'v2\'),\n                          \'weights\': reference_document[\'weights\']})\n    if args.stress_opponents:\n        reference.extend({\'kind\': \'bluffer\', \'bluff_rate\': b, \'call_rate\': c}\n                         for b, c in ((.1, .35), (.65, .35), (.95, .0), (.95, .35)))\n    archive = list(reference)\n    selection_seeds = list(range(args.selection_seed_start, args.selection_seed_start + args.validation_blocks))\n    history, started = [], time.time()\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers, mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        baseline = parallel_evaluate(pool, incumbent, selection_seeds, reference, \'league\')\n        initial = parallel_evaluate(pool, {\'kind\': args.policy_kind, \'weights\': best.tolist()}, selection_seeds, reference, \'league\')\n        selected_score = initial[\'points\']\n        for generation in range(args.iterations):\n            if generation and generation % 10 == 0:\n                mean, sigma = best.copy(), np.full(len(mean), 1.5)\n            population = rng.normal(mean, sigma, (args.population, len(mean)))\n            probes = max(1, args.population // 5)\n            population[-probes:] = rng.normal(best, 3., (probes, len(mean)))\n            population = np.clip(population, -20., 20.)\n            population[0] = mean\n            if args.population > 1:\n                population[1] = best\n            seeds = list(range(args.train_seed_start + generation * args.train_blocks, args.train_seed_start + (generation + 1) * args.train_blocks))\n            # Fast training probes use a shorter cap; model selection and final\n            # reporting always use the real 1,000-turn cap. Short-cap results are\n            # approximate training rewards, never evidence of event performance.\n            scores = list(pool.map(evaluate_job, [({\'kind\':args.policy_kind,\'weights\':w.tolist()},seeds,archive,\'league\',args.train_horizon) for w in population]))\n            ranked = sorted(range(len(scores)), key=lambda i: (scores[i][\'points\'], scores[i][\'win_rate\']), reverse=True)\n            elite = population[ranked[:max(2, args.population // 5)]]\n            mean = .35 * mean + .65 * elite.mean(axis=0)\n            sigma = np.maximum(.3, .35 * sigma + .65 * elite.std(axis=0))\n            candidate = population[ranked[0]]\n            spec = {\'kind\':args.policy_kind,\'weights\':candidate.tolist()}\n            validation = parallel_evaluate(pool, spec, selection_seeds, reference, \'league\')\n            if validation[\'points\'] > selected_score:\n                best, selected_score = candidate.copy(), validation[\'points\']\n            if generation % 4 == 3:\n                archive.append(spec)\n                archive = reference + archive[len(reference):][-12:]\n            record = {\'generation\':generation+1,\'train\':summarize(scores[ranked[0]]),\'validation\':summarize(validation),\n                      \'paired_validation_gain\':paired_interval(validation,baseline),\'selected_points\':selected_score,\n                      \'archive_size\':len(archive),\'elapsed_seconds\':round(time.time()-started,1)}\n            history.append(record); print(json.dumps(record),flush=True)\n            policy_class = V5 if args.policy_kind == \'v5\' else V4 if args.policy_kind == \'v4\' else V2\n            policy_class(best).save(args.output/\'bluff-policy-v2.json\',{\'method\':\'archive-league cross-entropy policy optimization\',\n                \'fast_observations\': args.fast_observations, \'stress_opponents\': args.stress_opponents,\n                \'policy_kind\': args.policy_kind, \'selection_seed_start\': args.selection_seed_start,\n                \'test_seed_start\': args.test_seed_start, \'train_seed_start\': args.train_seed_start,\n                \'seed\':args.seed,\'generations\':generation+1,\'population\':args.population,\n                \'train_blocks\':args.train_blocks,\'train_horizon\':args.train_horizon,\'evaluation_horizon\':1000,\'validation_blocks\':args.validation_blocks,\n                \'selected_validation_points\':selected_score,\'incumbent_validation_points\':baseline[\'points\'],\n                \'sdk_commit\':\'96da8042c285af4e4f43655f2ad99592bd49b8e6\'})\n            (args.output/\'training-progress.json\').write_text(json.dumps(history,indent=2)+\'\\n\')\n            (args.output/\'archive.json\').write_text(json.dumps(archive,indent=2)+\'\\n\')\n        seeds = list(range(args.test_seed_start,args.test_seed_start+args.test_blocks))\n        chosen = {\'kind\':args.policy_kind,\'weights\':best.tolist()}\n        modes = (\'league\',\'starter\',\'truthful\',\'bluffer\',\'aggressive\',\'competitive\')\n        results = list(pool.map(evaluate_job,[(spec,seeds,reference,mode) for mode in modes for spec in (chosen,incumbent)]))\n        report = {mode:{\'candidate\':summarize(results[2*i]),\'incumbent\':summarize(results[2*i+1]),\n                        \'paired_gain\':paired_interval(results[2*i],results[2*i+1])} for i,mode in enumerate(modes)}\n        (args.output/\'evaluation.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n        print(json.dumps({\'held_out_evaluation\':report}),flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'train_bluff_residual.py': '"""GPU fit of paired rollout preferences; actual-game tests gate improvement."""\nimport argparse\nimport concurrent.futures\nimport copy\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport time\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom bluff_policy_v5 import FEATURE_NAMES\nfrom bluff_mlp_policy import Policy\nfrom train_bluff_league import evaluate_job, paired_interval, summarize\n\n\nclass Residual(nn.Module):\n    def __init__(self, base, mean, scale, width):\n        super().__init__()\n        self.register_buffer(\'base\', torch.tensor(base, dtype=torch.float32))\n        self.register_buffer(\'mean\', torch.tensor(mean, dtype=torch.float32))\n        self.register_buffer(\'scale\', torch.tensor(scale, dtype=torch.float32))\n        self.hidden = nn.Linear(len(base), width)\n        self.output = nn.Linear(width, 1)\n        nn.init.zeros_(self.output.weight); nn.init.zeros_(self.output.bias)\n\n    def forward(self, features):\n        residual = self.output(F.relu(self.hidden((features - self.mean) / self.scale))).squeeze(-1)\n        return features @ self.base + residual, residual\n\n\ndef pairs(data, seed):\n    ids = np.unique(data[\'game_ids\']); rng = np.random.default_rng(seed); rng.shuffle(ids)\n    validation_ids = set(ids[:max(1, len(ids) // 5)].tolist())\n    train, validation, training_rows = [], [], []\n    for state, (start, stop) in enumerate(zip(data[\'offsets\'][:-1], data[\'offsets\'][1:])):\n        is_validation = int(data[\'game_ids\'][state]) in validation_ids\n        destination = validation if is_validation else train\n        if not is_validation:\n            training_rows.extend(range(start, stop))\n        for first in range(start, stop):\n            for second in range(first + 1, stop):\n                gap = float(data[\'targets\'][first] - data[\'targets\'][second])\n                if np.array_equal(data[\'features\'][first], data[\'features\'][second]):\n                    continue\n                differences = data[\'trial_targets\'][first] - data[\'trial_targets\'][second]\n                stderr = float(np.std(differences, ddof=1) / np.sqrt(len(differences))) if len(differences) > 1 else 0.\n                ranking_weight = min(1., abs(gap) / 3.) * max(.05, 1. - stderr / abs(gap)) if abs(gap) >= .25 else 0.\n                # Keep zero-gap observations for value regression. Omitting\n                # them would condition each pair on a different nonzero event.\n                value_weight = 1. / (1. + stderr ** 2)\n                destination.append((first, second, 1. / (1. + np.exp(-gap / .75)), ranking_weight, gap, value_weight))\n    if not train or not validation or not any(row[3] > 0. for row in train) or not any(row[3] > 0. for row in validation):\n        raise ValueError(\'not enough informative whole-game-held-out pairs\')\n    return np.asarray(train), np.asarray(validation), np.asarray(training_rows), len(ids) - len(validation_ids), len(validation_ids)\n\n\ndef loss(model, features, comparisons, penalty, kind):\n    a = comparisons[:, 0].long(); b = comparisons[:, 1].long()\n    score_a, residual_a = model(features[a]); score_b, residual_b = model(features[b])\n    if kind == \'ranking\':\n        point = F.binary_cross_entropy_with_logits(score_a - score_b, comparisons[:, 2], reduction=\'none\')\n        weight = comparisons[:, 3]\n    else:\n        point = (score_a - score_b - comparisons[:, 4]).square()\n        weight = comparisons[:, 5]\n    preference = (point * weight).sum() / weight.sum().clamp_min(1e-8)\n    return preference + penalty * (residual_a.square().mean() + residual_b.square().mean()) / 2., preference\n\n\ndef export(model, metadata):\n    return {\'format\': \'triad-bluff-residual-mlp-v1\', \'features\': FEATURE_NAMES,\n            \'base_weights\': model.base.detach().cpu().tolist(),\n            \'feature_mean\': model.mean.detach().cpu().tolist(), \'feature_scale\': model.scale.detach().cpu().tolist(),\n            \'hidden_weight\': model.hidden.weight.detach().cpu().tolist(),\n            \'hidden_bias\': model.hidden.bias.detach().cpu().tolist(),\n            \'output_weight\': model.output.weight.detach().cpu().flatten().tolist(),\n            \'output_bias\': float(model.output.bias.detach().cpu()[0]), \'training\': metadata}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--data\', type=Path, required=True)\n    parser.add_argument(\'--previous\', type=Path, required=True)\n    parser.add_argument(\'--width\', type=int, default=64)\n    parser.add_argument(\'--epochs\', type=int, default=120)\n    parser.add_argument(\'--batch\', type=int, default=4096)\n    parser.add_argument(\'--workers\', type=int, default=4)\n    parser.add_argument(\'--validation-blocks\', type=int, default=128)\n    parser.add_argument(\'--test-blocks\', type=int, default=256)\n    parser.add_argument(\'--validation-seed-start\', type=int, default=51_000_000)\n    parser.add_argument(\'--test-seed-start\', type=int, default=52_000_000)\n    parser.add_argument(\'--seed\', type=int, default=20261020)\n    parser.add_argument(\'--device\', default=\'cuda\')\n    parser.add_argument(\'--loss-types\', default=\'ranking,value\')\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args()\n    loss_types = args.loss_types.split(\',\')\n    if not loss_types or len(set(loss_types)) != len(loss_types) or any(kind not in (\'ranking\', \'value\') for kind in loss_types):\n        parser.error(\'loss types must be distinct ranking/value choices\')\n    if min(args.width, args.epochs, args.batch, args.workers, args.validation_blocks, args.test_blocks) < 1 or args.width > 128:\n        parser.error(\'positive counts and width<=128 required\')\n    if args.device == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'authorized GPU unavailable; no paid or hardware fallback\')\n    torch.set_num_threads(1)\n    os.environ[\'BLUFF_FAST_OBSERVATIONS\'] = \'1\'\n    os.environ[\'OPENBLAS_NUM_THREADS\'] = \'1\'\n    args.output.mkdir(parents=True, exist_ok=True)\n    metadata = json.loads((args.data.parent / \'metadata.json\').read_text())\n    if metadata[\'input_features\'] != FEATURE_NAMES:\n        raise ValueError(\'counterfactual schema differs from deployed feature extractor\')\n    with np.load(args.data) as archive:\n        data = {name: archive[name] for name in archive.files}\n    if data[\'features\'].shape[1] != len(FEATURE_NAMES) or not np.isfinite(data[\'features\']).all():\n        raise ValueError(\'invalid training matrix\')\n    training, validation, training_rows, training_games, validation_games = pairs(data, args.seed)\n    mean = data[\'features\'][training_rows].mean(axis=0)\n    scale = np.maximum(.1, data[\'features\'][training_rows].std(axis=0))\n    # Positive quarter scaling preserves linear move ranking while giving the\n    # residual a manageable score scale; only neural parameters are optimized.\n    base = [weight / 4. for weight in metadata[\'incumbent\'][\'weights\']]\n    base += [0.] * (len(FEATURE_NAMES) - len(base))\n    features = torch.as_tensor(data[\'features\'], dtype=torch.float32, device=args.device)\n    training = torch.as_tensor(training, dtype=torch.float32, device=args.device)\n    validation = torch.as_tensor(validation, dtype=torch.float32, device=args.device)\n    print(json.dumps({\'device\': args.device, \'gpu_count\': torch.cuda.device_count(), \'states\': len(data[\'game_ids\']),\n                      \'actions\': len(features), \'training_games\': training_games, \'validation_games\': validation_games,\n                      \'training_pairs\': len(training), \'validation_pairs\': len(validation)}), flush=True)\n    candidates, history = [], []; started = time.time()\n    for index, (loss_type, penalty) in enumerate((kind, penalty) for kind in loss_types for penalty in (.01, .03, .1)):\n        torch.manual_seed(args.seed + index)\n        model = Residual(base, mean, scale, args.width).to(args.device)\n        optimizer = torch.optim.AdamW(model.parameters(), lr=.003, weight_decay=.0001)\n        with torch.no_grad():\n            _, initial_loss = loss(model, features, validation, penalty, loss_type)\n        best_loss, best_epoch, best = float(initial_loss), 0, copy.deepcopy(model.state_dict())\n        for epoch in range(args.epochs):\n            order = torch.randperm(len(training), device=args.device)\n            for start in range(0, len(order), args.batch):\n                batch = training[order[start:start + args.batch]]\n                objective, _ = loss(model, features, batch, penalty, loss_type)\n                optimizer.zero_grad(set_to_none=True); objective.backward()\n                nn.utils.clip_grad_norm_(model.parameters(), 5.); optimizer.step()\n            with torch.no_grad():\n                _, current = loss(model, features, validation, penalty, loss_type)\n            if float(current) < best_loss:\n                best_loss, best_epoch, best = float(current), epoch + 1, copy.deepcopy(model.state_dict())\n            if (epoch + 1) % 10 == 0:\n                row = {\'loss_type\': loss_type, \'regularization\': penalty, \'epoch\': epoch + 1, \'validation_preference_loss\': float(current),\n                       \'best_preference_loss\': best_loss, \'seconds\': round(time.time() - started, 1)}\n                history.append(row); print(json.dumps(row), flush=True)\n        model.load_state_dict(best)\n        document = export(model, {\'method\': \'paired counterfactual residual policy distillation\',\n            \'seed\': args.seed + index, \'loss_type\': loss_type, \'regularization\': penalty, \'best_epoch\': best_epoch,\n            \'validation_preference_loss\': best_loss, \'baseline_preference_loss\': float(initial_loss),\n            \'training_games\': training_games, \'validation_games\': validation_games, \'source_dataset\': metadata,\n            \'base_score_factor\': .25, \'hidden_state_use\': \'training labels only; policy input is own/public features\'})\n        check = data[\'features\'][:min(200, len(features))]\n        with torch.no_grad():\n            tensor = torch.as_tensor(check, dtype=torch.float32, device=args.device)\n            _, residual = model(tensor); base_scores = tensor @ model.base\n        for strength in (.25, .5, 1.):\n            scaled = copy.deepcopy(document)\n            scaled[\'output_weight\'] = [weight * strength for weight in scaled[\'output_weight\']]\n            scaled[\'output_bias\'] *= strength; scaled[\'training\'][\'residual_scale\'] = strength\n            deployed = Policy(scaled)\n            actual = np.asarray([deployed.score_features(row.tolist()) for row in check])\n            expected = (base_scores + strength * residual).cpu().numpy()\n            maximum = float(np.max(np.abs(actual - expected)))\n            if maximum > 1e-4:\n                raise RuntimeError(\'standard-library neural export differs from training network\')\n            path = args.output / f\'candidate-{len(candidates)}.json\'; path.write_text(json.dumps(scaled, indent=2) + \'\\n\')\n            print(json.dumps({\'candidate\': len(candidates), \'export_max_score_error\': maximum}), flush=True)\n            candidates.append(scaled)\n    (args.output / \'training-progress.json\').write_text(json.dumps(history, indent=2) + \'\\n\')\n    incumbent = {\'kind\': \'v2\', \'weights\': metadata[\'incumbent\'][\'weights\']}\n    previous = json.loads(args.previous.read_text())\n    reference = [{\'kind\': kind} for kind in (\'starter\', \'competitive\', \'heuristic\', \'random\', \'truthful\', \'aggressive\')]\n    reference += [incumbent, {\'kind\': \'v1\', \'weights\': previous[\'weights\']}]\n    reference += [{\'kind\': \'bluffer\', \'bluff_rate\': bluff, \'call_rate\': call}\n                  for bluff, call in ((.1, .0), (.1, .35), (.35, .1), (.65, .35), (.95, .0), (.95, .35))]\n    specs = [incumbent] + [{\'kind\': \'mlp\', \'model\': document} for document in candidates]\n    selection_seeds = list(range(args.validation_seed_start, args.validation_seed_start + args.validation_blocks))\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers, mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        results = list(pool.map(evaluate_job, [(spec, selection_seeds, reference, \'league\') for spec in specs]))\n        chosen = max(range(len(results)), key=lambda index: results[index][\'points\'])\n        selection = {\'selected\': chosen, \'seed_start\': args.validation_seed_start,\n                     \'scores\': [summarize(result) for result in results],\n                     \'paired_gains\': [paired_interval(result, results[0]) for result in results[1:]]}\n        (args.output / \'selection.json\').write_text(json.dumps(selection, indent=2) + \'\\n\')\n        print(json.dumps({\'actual_game_selection\': selection}), flush=True)\n        selected = specs[chosen]\n        if chosen:\n            (args.output / \'bluff-policy-mlp.json\').write_text(json.dumps(candidates[chosen - 1], indent=2) + \'\\n\')\n        else:\n            (args.output / \'retained-incumbent.json\').write_text(json.dumps(metadata[\'incumbent\'], indent=2) + \'\\n\')\n        final_seeds = list(range(args.test_seed_start, args.test_seed_start + args.test_blocks))\n        modes = (\'league\', \'competitive\', \'starter\', \'truthful\', \'bluffer\', \'aggressive\')\n        final = list(pool.map(evaluate_job, [(spec, final_seeds, reference, mode) for mode in modes for spec in (selected, incumbent)]))\n        report = {mode: {\'candidate\': summarize(final[2 * index]), \'incumbent\': summarize(final[2 * index + 1]),\n                         \'paired_gain\': paired_interval(final[2 * index], final[2 * index + 1])} for index, mode in enumerate(modes)}\n        (args.output / \'evaluation.json\').write_text(json.dumps({\'seed_start\': args.test_seed_start, \'evaluation\': report}, indent=2) + \'\\n\')\n        print(json.dumps({\'held_out_actual_games\': report}), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'}
for name,source in sources.items(): (work/name).write_text(source)
(work/"previous.json").write_text('{\n  "format": "triad-bluff-linear-v1",\n  "features": [\n    "batch_size",\n    "truthful",\n    "bluff",\n    "bluff_pile",\n    "bluff_call_rate",\n    "bluff_finish",\n    "remaining_groups",\n    "remaining_largest_group",\n    "claimed_rank_held",\n    "bias",\n    "revealed_bluff_rate",\n    "rank_held",\n    "claim_count",\n    "pile_size",\n    "opponent_finishing",\n    "own_small_hand",\n    "opponent_small_hand",\n    "samples"\n  ],\n  "weights": [\n    4.351868833686492,\n    5.6322712663248415,\n    -2.6260873498626167,\n    0.7063932700439369,\n    -10.84185815552825,\n    -8.300750779890834,\n    -0.7486711333834275,\n    1.2260583122844886,\n    -1.5575700886723332,\n    -4.558178606791033,\n    8.25750430388175,\n    0.43873901259948533,\n    -1.3556124209301408,\n    -1.1333529834957075,\n    0.14766041336362645,\n    -1.5734330345566885,\n    -0.4730921181099686,\n    -0.5811911507935986\n  ],\n  "training": {\n    "method": "cross-entropy policy optimization",\n    "seed": 20261014,\n    "generations": 25,\n    "population": 32,\n    "games_per_candidate": 64,\n    "validation": 3.86328125,\n    "sdk_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6"\n  }\n}\n')
sdk=Path("/kaggle/working/bluffSDK")
subprocess.run(["git","clone","--quiet","https://github.com/Sir-Nike/bluffSDK",str(sdk)],check=True)
subprocess.run(["git","-C",str(sdk),"checkout","--quiet","96da8042c285af4e4f43655f2ad99592bd49b8e6"],check=True)
os.environ["BLUFF_SDK"]=str(sdk)
data=next(Path("/kaggle/input").rglob("counterfactual.npz"))
for command,timeout in [(['python', '/kaggle/working/training/train_bluff_residual.py', '--data', 'DATA_PLACEHOLDER', '--previous', '/kaggle/working/training/previous.json', '--width', '64', '--batch', '4096', '--workers', '4', '--device', 'cuda', '--epochs', '3', '--validation-blocks', '2', '--test-blocks', '2', '--validation-seed-start', '61000000', '--test-seed-start', '62000000', '--output', '/kaggle/working/smoke'], 3600), (['python', '/kaggle/working/training/train_bluff_residual.py', '--data', 'DATA_PLACEHOLDER', '--previous', '/kaggle/working/training/previous.json', '--width', '64', '--batch', '4096', '--workers', '4', '--device', 'cuda', '--epochs', '120', '--validation-blocks', '128', '--test-blocks', '256', '--validation-seed-start', '51000000', '--test-seed-start', '52000000', '--output', '/kaggle/working/output'], 10800)]:
    command=[str(data) if item=="DATA_PLACEHOLDER" else item for item in command]
    subprocess.run(command,check=True,timeout=timeout)
